In [ ]:
import subprocess
import shutil
import pandas as pd
from pathlib import Path

In [ ]:
# selectie_gebied = 0 # Oude IJssel
# selectie_gebied = 1 # West
# selectie_gebied = 2 # Centraal
# selectie_gebied = 3 # Oost

selectie_gebieden = [3]

scenarios = ["REF", "SCEN"]

start_date = "2010-4-1"
# end_date = "2018-12-31"
end_date = "2010-4-12"

# path to the package containing the dummy-data
dir_model_basis = "..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel\\"

In [ ]:
simulaties_total = pd.DataFrame()

for selectie_gebied in selectie_gebieden:
    for scenario in scenarios:
        # date_range = pd.date_range(start_date, end_date, freq="6MS")
        date_range = pd.date_range(start_date, end_date, freq="2D")

        simulaties = pd.DataFrame()
        simulaties["start_date"] = date_range
        simulaties["end_date"] = simulaties["start_date"].shift(-1)
        simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
        simulaties["seizoen"] = ["zomer", "winter"]*int(len(date_range)/2)
        simulaties["scenario"] = scenario
        simulaties["gebied"] = selectie_gebied
        simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

        simulaties["model_name"] = simulaties.apply(lambda x: f"rr_model__{pd.to_datetime(x.start_date).strftime('%Y_%m_%d')}__{pd.to_datetime(x.end_date).strftime('%Y_%m_%d')}", axis=1)
        simulaties_total = pd.concat([simulaties_total, simulaties])

In [ ]:
simulaties_total

In [ ]:
for index, simulatie in simulaties_total.iloc[:2].iterrows():
    print(str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

    dir_model = Path(dir_model_basis, f"gebied_{simulatie.gebied}", simulatie.scenario)

    if index > 0:
        unpaved_restart_out = Path(dir_model, simulaties.loc[index-1, "model_name"], "rr", "RSRR_OUT")
        unpaved_restart_in = Path(dir_model, simulaties.loc[index, "model_name"], "rr", "RSRR_IN")
        shutil.copy(unpaved_restart_out, unpaved_restart_in)
    
    subprocess.run(
        ["cmd.exe", "/c", "run.bat"],
        cwd=str(Path(dir_model, simulaties.loc[index, "model_name"]))
    )